In [1]:
import numpy as np
import matplotlib.pyplot as plt
import imodex_miri_postprocess_pipeline as imp #homemade pipeline
from scipy import ndimage
from scipy.optimize import curve_fit
from astropy.stats import sigma_clip
from astropy.io import fits
import glob
from scipy.optimize import minimize
#from collections import defaultdict
from scipy import ndimage
import math
import glob, os
import scipy.optimize as optimize
from matplotlib.colors import LogNorm

#ploting
import matplotlib.animation as animation
from matplotlib.animation import PillowWriter

# Réduction de Kappa And - GO 01241

In [4]:
################
#### F1140C ####
################



path_save = '/home/cperrot/Work/projet/JWST/Target/GTO/P01241/Kappa_And/'

if not os.path.exists(path_save):
    os.makedirs(path_save)

pixscale = 0.110 # arcsec
lam = 11.40e-6    # lambda en m


aff  = 'cal'

path_tar  = '/home/cperrot/Work/raw/JWST/Target/MAST/CAL/P01241/obs017/'
path_ref  = '/home/cperrot/Work/raw/JWST/Target/MAST/CAL/P01241/obs020/'


path_bkg_1  = '/home/cperrot/Work/raw/JWST/Target/MAST/MASTER_BKG/P01241/obs014/'
path_bkg_2  = '/home/cperrot/Work/raw/JWST/Target/MAST/MASTER_BKG/P01241/obs023/'

cube_tar ,header_tar   = imp.load_cube(path_tar,aff)
cube_ref ,header_ref   = imp.load_cube(path_ref,aff)

min_bkg_1 = fits.open(glob.glob(path_bkg_1+'P*_master_bkg_cal.fits',recursive=False)[0])[0].data
min_bkg_2 = fits.open(glob.glob(path_bkg_2+'P*_master_bkg_cal.fits',recursive=False)[0])[0].data
        
#bkg subtraction
cube_tar = cube_tar-min_bkg_1
cube_ref = cube_ref-min_bkg_2

clipval = 3.
cube_tar  = imp.sigma_clip_jwst(cube_tar,clipval)
cube_ref  = imp.sigma_clip_jwst(cube_ref,clipval)

#resize
newd = 200
xd = np.shape(cube_tar)[1]
yd = np.shape(cube_tar)[0]
x0 = 119
y0 = 112
scale=[(newd/2.)*pixscale,-1*(newd/2.)*pixscale,-1*(newd/2.)*pixscale,(newd/2.)*pixscale]

cube_tar = cube_tar[y0-newd//2:y0+newd//2,x0-newd//2:x0+newd//2]
cube_ref = cube_ref[:,y0-newd//2:y0+newd//2,x0-newd//2:x0+newd//2]

#PCA
PCA_1140 = imp.sub_PCA(imp.clean_image_nan(cube_tar), imp.clean_image_nan(cube_ref), [newd//2,newd//2], 3, dim=newd//2, plot=False)

# RDI
f_scale = imp.flux_scaling_ta(path_tar+'TA', path_ref+'TA', aff, lam)
print("f_scale with TA : ",f_scale)

#perform RDI with 1 roll and 1 reference
RDI_1140 = imp.RDI_classic(cube_tar, cube_ref, f_scale=f_scale)

#rotation
PA_V3 = header_tar[1]['PA_V3']     # 59.58852013851359
V3i   = header_tar[1]['V3I_YANG']  # 4.83544897

PCA_1140_r = ndimage.rotate(PCA_1140, -(PA_V3+V3i), reshape=False)
RDI_1140_c = np.copy(RDI_1140)
RDI_1140_c = imp.clean_image_nan(RDI_1140_c)
RDI_1140_r = ndimage.rotate(RDI_1140_c, -(PA_V3+V3i), reshape=False)


imp.save_fits(RDI_1140,path_save+'RDI_1140.fits')
imp.save_fits(RDI_1140_r,path_save+'RDI_1140_rot.fits')

imp.save_fits(PCA_1140,path_save+'PCA_1140.fits')
imp.save_fits(PCA_1140_r,path_save+'PCA_1140_rot.fits')

imp.save_fits(cube_tar,path_save+'cube_1140_pixelcorr.fits')

imp.save_fits(RDI_1140_c,path_save+'RDI_1140_clean.fits')
imp.save_fits(imp.clean_image_nan(cube_tar),path_save+'cube_1140_pixelcorr_clean.fits')



Loading of 1 cubes.
Number of dimension : 2
Initial images size (px) : 288 / 224
New images size (px)     : 288 / 224
Loading of 5 cubes.
Number of dimension : 2
Initial images size (px) : 288 / 224
New images size (px)     : 288 / 224
Loading of 2 cubes.
Number of dimension : 2
Initial images size (px) : 288 / 224
New images size (px)     : 200 / 200
sigma model PSF:  1.3965812874037857
155 155
Roll - TA 1
Result : A = 7379.8 / x0 = -21437.51 / y0 = 127.29 / width_x = 15210.57 / width_y = 130.19
Stddev : A = 2381255.9 / x0 = 7031550.04 / y0 = 0.93 / width_x = 2500840.43 / width_y = 1.72
115 117
Roll - TA 2
Result : A = 2860.3 / x0 = 1.03 / y0 = 124.82 / width_x = 701.19 / width_y = 131.13
Stddev : A = 17.0 / x0 = 65.82 / y0 = 0.83 / width_x = 229.03 / width_y = 1.62
Loading of 2 cubes.
Number of dimension : 2
Initial images size (px) : 288 / 224
New images size (px)     : 200 / 200
157 155
Ref - TA 1
Result : A = 17285.6 / x0 = 3290.73 / y0 = 149.42 / width_x = 1703.82 / width_y = 134

In [5]:
################
#### F1065C ####
################



path_save = '/home/cperrot/Work/projet/JWST/Target/GTO/P01241/Kappa_And/'

if not os.path.exists(path_save):
    os.makedirs(path_save)

pixscale = 0.110 # arcsec
lam = 10.65e-6    # lambda en m


aff  = 'cal'

path_tar  = '/home/cperrot/Work/raw/JWST/Target/MAST/CAL/P01241/obs018/'
path_ref  = '/home/cperrot/Work/raw/JWST/Target/MAST/CAL/P01241/obs019/'


path_bkg_1  = '/home/cperrot/Work/raw/JWST/Target/MAST/MASTER_BKG/P01241/obs015/'
path_bkg_2  = '/home/cperrot/Work/raw/JWST/Target/MAST/MASTER_BKG/P01241/obs022/'

cube_tar ,header_tar   = imp.load_cube(path_tar,aff)
cube_ref ,header_ref   = imp.load_cube(path_ref,aff)

min_bkg_1 = fits.open(glob.glob(path_bkg_1+'P*_master_bkg_cal.fits',recursive=False)[0])[0].data
min_bkg_2 = fits.open(glob.glob(path_bkg_2+'P*_master_bkg_cal.fits',recursive=False)[0])[0].data
        
#bkg subtraction
cube_tar = cube_tar-min_bkg_1
cube_ref = cube_ref-min_bkg_2

clipval = 3.
cube_tar  = imp.sigma_clip_jwst(cube_tar,clipval)
cube_ref  = imp.sigma_clip_jwst(cube_ref,clipval)

#resize
newd = 200
xd = np.shape(cube_tar)[1]
yd = np.shape(cube_tar)[0]
x0 = 119
y0 = 112
scale=[(newd/2.)*pixscale,-1*(newd/2.)*pixscale,-1*(newd/2.)*pixscale,(newd/2.)*pixscale]

cube_tar = cube_tar[y0-newd//2:y0+newd//2,x0-newd//2:x0+newd//2]
cube_ref = cube_ref[:,y0-newd//2:y0+newd//2,x0-newd//2:x0+newd//2]

#PCA
PCA_1065 = imp.sub_PCA(imp.clean_image_nan(cube_tar), imp.clean_image_nan(cube_ref), [newd//2,newd//2], 3, dim=newd//2, plot=False)

# RDI
f_scale = imp.flux_scaling_ta(path_tar+'TA', path_ref+'TA', aff, lam)
print("f_scale with TA : ",f_scale)

#perform RDI with 1 roll and 1 reference
RDI_1065 = imp.RDI_classic(cube_tar, cube_ref, f_scale=f_scale)

#rotation
PA_V3 = header_tar[1]['PA_V3']     # 59.58852013851359
V3i   = header_tar[1]['V3I_YANG']  # 4.83544897

PCA_1065_r = ndimage.rotate(PCA_1065, -(PA_V3+V3i), reshape=False)
RDI_1065_c = np.copy(RDI_1065)
RDI_1065_c = imp.clean_image_nan(RDI_1065_c)
RDI_1065_r = ndimage.rotate(RDI_1065_c, -(PA_V3+V3i), reshape=False)


imp.save_fits(RDI_1065,path_save+'RDI_1065.fits')
imp.save_fits(RDI_1065_r,path_save+'RDI_1065_rot.fits')

imp.save_fits(PCA_1065,path_save+'PCA_1065.fits')
imp.save_fits(PCA_1065_r,path_save+'PCA_1065_rot.fits')

imp.save_fits(cube_tar,path_save+'cube_1065_pixelcorr.fits')

imp.save_fits(RDI_1065_c,path_save+'RDI_1065_clean.fits')
imp.save_fits(imp.clean_image_nan(cube_tar),path_save+'cube_1065_pixelcorr_clean.fits')



Loading of 1 cubes.
Number of dimension : 2
Initial images size (px) : 288 / 224
New images size (px)     : 288 / 224
Loading of 5 cubes.
Number of dimension : 2
Initial images size (px) : 288 / 224
New images size (px)     : 288 / 224
Loading of 2 cubes.
Number of dimension : 2
Initial images size (px) : 288 / 224
New images size (px)     : 200 / 200
sigma model PSF:  1.3047009395482736
155 158
Roll - TA 1
Result : A = 20470.2 / x0 = -20406.48 / y0 = -85.16 / width_x = 10090.37 / width_y = 668.59
Stddev : A = 5751930.4 / x0 = 2790418.67 / y0 = 111.03 / width_x = 686557.87 / width_y = 200.23
113 115
Roll - TA 2
Result : A = 6420.1 / x0 = -5262.59 / y0 = -10.23 / width_x = 4055.89 / width_y = 346.51
Stddev : A = 116964.3 / x0 = 113451.59 / y0 = 17.82 / width_x = 43278.05 / width_y = 27.98
Loading of 2 cubes.
Number of dimension : 2
Initial images size (px) : 288 / 224
New images size (px)     : 200 / 200
156 158
Ref - TA 1
Result : A = 40258.2 / x0 = 6203.51 / y0 = 1215.26 / width_x = 2

In [6]:
################
#### F1550C ####
################



path_save = '/home/cperrot/Work/projet/JWST/Target/GTO/P01241/Kappa_And/'

if not os.path.exists(path_save):
    os.makedirs(path_save)

pixscale = 0.110 # arcsec
lam = 15.50e-6    # lambda en m


aff  = 'cal'

path_tar  = '/home/cperrot/Work/raw/JWST/Target/MAST/CAL/P01241/obs016/'
path_ref  = '/home/cperrot/Work/raw/JWST/Target/MAST/CAL/P01241/obs021/'


path_bkg_1  = '/home/cperrot/Work/raw/JWST/Target/MAST/MASTER_BKG/P01241/obs013/'
path_bkg_2  = '/home/cperrot/Work/raw/JWST/Target/MAST/MASTER_BKG/P01241/obs024/'

cube_tar ,header_tar   = imp.load_cube(path_tar,aff)
cube_ref ,header_ref   = imp.load_cube(path_ref,aff)

min_bkg_1 = fits.open(glob.glob(path_bkg_1+'P*_master_bkg_cal.fits',recursive=False)[0])[0].data
min_bkg_2 = fits.open(glob.glob(path_bkg_2+'P*_master_bkg_cal.fits',recursive=False)[0])[0].data
        
#bkg subtraction
cube_tar = cube_tar-min_bkg_1
cube_ref = cube_ref-min_bkg_2

clipval = 3.
cube_tar  = imp.sigma_clip_jwst(cube_tar,clipval)
cube_ref  = imp.sigma_clip_jwst(cube_ref,clipval)

#resize
newd = 200
xd = np.shape(cube_tar)[1]
yd = np.shape(cube_tar)[0]
x0 = 119
y0 = 112
scale=[(newd/2.)*pixscale,-1*(newd/2.)*pixscale,-1*(newd/2.)*pixscale,(newd/2.)*pixscale]

cube_tar = cube_tar[y0-newd//2:y0+newd//2,x0-newd//2:x0+newd//2]
cube_ref = cube_ref[:,y0-newd//2:y0+newd//2,x0-newd//2:x0+newd//2]

#PCA
PCA_1550 = imp.sub_PCA(imp.clean_image_nan(cube_tar), imp.clean_image_nan(cube_ref), [newd//2,newd//2], 2, dim=newd//2, plot=False)

# RDI
f_scale = imp.flux_scaling_ta(path_tar+'TA', path_ref+'TA', aff, lam)
print("f_scale with TA : ",f_scale)

#perform RDI with 1 roll and 1 reference
RDI_1550 = imp.RDI_classic(cube_tar, cube_ref, f_scale=f_scale)

#rotation
PA_V3 = header_tar[1]['PA_V3']     # 59.58852013851359
V3i   = header_tar[1]['V3I_YANG']  # 4.83544897

PCA_1550_r = ndimage.rotate(PCA_1550, -(PA_V3+V3i), reshape=False)
RDI_1550_c = np.copy(RDI_1550)
RDI_1550_c = imp.clean_image_nan(RDI_1550_c)
RDI_1550_r = ndimage.rotate(RDI_1550_c, -(PA_V3+V3i), reshape=False)


imp.save_fits(RDI_1550,path_save+'RDI_1550.fits')
imp.save_fits(RDI_1550_r,path_save+'RDI_1550_rot.fits')

imp.save_fits(PCA_1550,path_save+'PCA_1065.fits')
imp.save_fits(PCA_1550_r,path_save+'PCA_1550_rot.fits')

imp.save_fits(cube_tar,path_save+'cube_1550_pixelcorr.fits')

imp.save_fits(RDI_1550_c,path_save+'RDI_1550_clean.fits')
imp.save_fits(imp.clean_image_nan(cube_tar),path_save+'cube_1550_pixelcorr_clean.fits')



Loading of 1 cubes.
Number of dimension : 2
Initial images size (px) : 288 / 224
New images size (px)     : 288 / 224
Loading of 5 cubes.
Number of dimension : 2
Initial images size (px) : 288 / 224
New images size (px)     : 288 / 224
Loading of 2 cubes.
Number of dimension : 2
Initial images size (px) : 288 / 224
New images size (px)     : 200 / 200
sigma model PSF:  1.8988605223472528
151 155
Roll - TA 1
Result : A = 36265.8 / x0 = 155.28 / y0 = 151.23 / width_x = 1.56 / width_y = 1.53
Stddev : A = 213.6 / x0 = 0.01 / y0 = 0.01 / width_x = 0.01 / width_y = 0.01
112 113
Roll - TA 2
Result : A = 36412.1 / x0 = 112.58 / y0 = 112.49 / width_x = 1.55 / width_y = 1.54
Stddev : A = 204.6 / x0 = 0.01 / y0 = 0.01 / width_x = 0.01 / width_y = 0.01
Loading of 2 cubes.
Number of dimension : 2
Initial images size (px) : 288 / 224
New images size (px)     : 200 / 200
153 155
Ref - TA 1
Result : A = 218739.6 / x0 = 155.32 / y0 = 152.76 / width_x = 1.57 / width_y = 1.56
Stddev : A = 276.0 / x0 = 0.